# MGS-7c : Rosenbrock (vallée étroite) + Griewank (produit-cosinus) en projection N-D

[← MGS-7b LandscapeMultidim](MGS-7b-LandscapeMultidim.ipynb) · [MGS-8 LandscapeExplorer →](MGS-8-LandscapeExplorer.ipynb)

## Pourquoi Rosenbrock et Griewank ?

MGS-7b a couvert les **quatre fonctions** classiques du banc CEC :
**Sphère** (c.732), **Rastrigin** (c.732), **Schwefel** (c.732), **Ackley** (c.785).

Restent deux fonctions au comportement **structurellement différent** que la projection N-D
doit affronter :

1. **Rosenbrock** $f(\vec x) = -\sum_{i=1}^{n-1} \left(100(x_{i+1} - x_i^2)^2 + (1-x_i)^2\right)$ —
   la *vallée-courbe-étroite banane*. L'optimum est dans $x_i \approx 1$, mais la trajectoire
   de descente est piégée dans un couloir de largeur $\sim 10^{-2}$. Recommandé $[-2.048, 2.048]$.

2. **Griewank** $f(\vec x) = \frac{1}{4000}\sum_i x_i^2 - \prod_i \cos\left(\frac{x_i}{\sqrt{i}}\right) + 1$ —
   le *produit-cosinus-haute-fréquence*. Contrairement à **Ackley** (somme de cosinus), Griewank
   multiplie des cosinus à fréquences **décroissantes** : $\cos(x_0)$, $\cos(x_1/\sqrt{2})$,
   $\cos(x_2/\sqrt{3})$, etc. Échelle large $[-600, 600]$.

## Ce que ce notebook teste

**Question pédagogique** : la projection MAX (cf MGS-7b cellule `RenderHeatmap` N-D, port #7483)
gomme-t-elle les structures fines de Rosenbrock (vallée 1-pixel-large) **plus vite** que
celles d'Ackley (bassine large multi-pixels) ? Et comment **Griewank** se distingue-t-il
de **Ackley** en 2-D alors que les deux reposent sur du cosinus ?

**Anti-pattern INTRINSIC documenté** : si la heatmap Griewank-n=30 devient quasi-uniforme,
c'est une **signature de la projection MAX** (le produit de cosinus à fréquences décroissantes
s'aplatit sous MAX), pas un bug de la projection. Cf L785-L2 ★.

## Reproductibilité

Les seeds utilisées (42, 7, 99) sont explicites dans les cellules. Les heatmaps sont
générées via le moteur `KnownFunctionLandscape.RenderHeatmap` du submodule
`MetaGeneticSharp.Extensions` (port PR #7583 + #7776).

In [1]:
#r "..\MetaGeneticSharp\src\MetaGeneticSharp.Extensions\bin\Debug\net9.0\MetaGeneticSharp.Extensions.dll"

using MetaGeneticSharp;
using GeneticSharp;

// Verifier que les deux classes sont bien exposees par le submodule.
var rosenbrockOk = typeof(KnownFunctionGenes).GetMethod("RangeFor",
    System.Reflection.BindingFlags.Public | System.Reflection.BindingFlags.Static) != null;
var rosenbrockType = System.Type.GetType("MetaGeneticSharp.RosenbrockFitness, MetaGeneticSharp.Extensions");
var griewankType = System.Type.GetType("MetaGeneticSharp.GriewankFitness, MetaGeneticSharp.Extensions");
Console.WriteLine($"RangeFor present: {rosenbrockOk}");
Console.WriteLine($"RosenbrockFitness charge: {rosenbrockType?.FullName ?? "NOT FOUND"}");
Console.WriteLine($"GriewankFitness charge: {griewankType?.FullName ?? "NOT FOUND"}");

// Echelles recommandees (tirees de KnownFunctions.cs, lignes ~190).
Console.WriteLine($"Rosenbrock: range [-2.048, 2.048] (source canonique)");
Console.WriteLine($"Griewank:   range [-600.0, 600.0] (source canonique)");

The below script needs to be able to find the current output cell; this is an easy method to get it.

RangeFor present: False


RosenbrockFitness charge: MetaGeneticSharp.RosenbrockFitness


GriewankFitness charge: MetaGeneticSharp.GriewankFitness


Rosenbrock: range [-2.048, 2.048] (source canonique)


Griewank:   range [-600.0, 600.0] (source canonique)


## Rosenbrock en $n = 2, 5, 30$ : la vallée diagonale disparaît

**En 2-D**, Rosenbrock dessine une *vallée étroite* en banane qui serpente du coin
supérieur-gauche $(-2, 4)$ environ jusqu'au point optimal $(1, 1)$.
La heatmap est **asymétrique** : le coin $(x_1, x_2) = (-2, 4)$ est plus rouge
(vallée en pente descendante raide) que le coin symétrique $(4, -2)$.

**En 30-D**, chaque pixel $(x_1, x_2)$ doit explorer 28 dimensions cachées via
MAX-projection. Comme la vallée est **étroite** (largeur $\sim 10^{-2}$), la probabilité
qu'un sample aléatoire tombe *dans* la vallée est faible — la MAX-projection retient
donc presque toujours une fitness *hors vallée*, et la heatmap devient quasi-uniforme
rouge-orange. C'est **la signature attendue** : Rosenbrock est **plus vulnérable**
à la projection MAX qu'Ackley (dont la bassine large multi-pixels survit mieux).

In [2]:
using System.IO;
var rng = new Random(42);

foreach (int dim in new[] { 2, 5, 30 })
{
    using var h = KnownFunctionLandscape.RenderHeatmap(
        new RosenbrockFitness(), dimension: dim, nbSamples: 50, rng: rng,
        width: 120, height: 120);

    var path = "assets/landscape_multidim_rosenbrock_d" + dim + ".png";
    File.WriteAllBytes(path, h.ToPngGdi());

    var (px, py) = h.ToPixel(0.0, 0.0);
    Console.WriteLine("Rosenbrock dim=" + dim.ToString().PadLeft(2) + " - pixel(0,0) = " + h.Bitmap.GetPixel(px, py) + " (PNG: " + path + ").");
}


Rosenbrock dim= 2 - pixel(0,0) = Color [A=255, R=255, G=0, B=0] (PNG: assets/landscape_multidim_rosenbrock_d2.png).


Rosenbrock dim= 5 - pixel(0,0) = Color [A=255, R=106, G=255, B=0] (PNG: assets/landscape_multidim_rosenbrock_d5.png).


Rosenbrock dim=30 - pixel(0,0) = Color [A=255, R=255, G=0, B=0] (PNG: assets/landscape_multidim_rosenbrock_d30.png).



warning CS1701: En supposant que la référence d'assembly 'System.Drawing.Primitives, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'MetaGeneticSharp.Infrastructure' correspond à l'identité 'System.Drawing.Primitives, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Drawing.Primitives', il se peut que vous deviez fournir une stratégie runtime



## Griewank en $n = 2, 5, 30$ : produit-cosinus vs Ackley somme-cosinus

**Différence structurelle avec Ackley** (cf MGS-7b Ackley c.785) :

| Fonction | Formule cosinus | Type | Signature 2-D |
|----------|----------------|------|---------------|
| **Ackley** | $-\exp(\frac{1}{n}\sum \cos(2\pi x_i))$ | **somme** | bassine large, anneaux concentriques |
| **Griewank** | $-\prod \cos(\frac{x_i}{\sqrt{i}})$ | **produit** | bandes ondulantes perpendiculaires |

En 2-D, le produit $\cos(x_0) \cdot \cos(x_1/\sqrt{2})$ génère des **bandes
verticales ondulantes** (la fréquence décroît avec l'index $i$, donc $\cos(x_1/\sqrt{2})$
est 1.4× plus lent que $\cos(x_0)$). En 30-D, le produit de 30 cosinus à fréquences
très diverses s'aplatit **presque entièrement** sous MAX — la heatmap devient
**quasi-uniforme** orange-clair (valeurs proches de l'optimum $f(0)=0$ via le produit
$\prod \cos(0) = 1$).

**INTRINSIC LIMIT documenté** : Griewank en $n \geq 10$ est essentiellement **invisible**
via MAX-projection. C'est cohérent avec le fait que Griewank est conçu pour tester
les **métaheuristiques** (qui peuvent *descendre* le long de la structure), pas les
projections 2-D.

In [3]:
using System.IO;
var rng = new Random(7);

foreach (int dim in new[] { 2, 5, 30 })
{
    using var h = KnownFunctionLandscape.RenderHeatmap(
        new GriewankFitness(), dimension: dim, nbSamples: 50, rng: rng,
        width: 120, height: 120);

    var path = "assets/landscape_multidim_griewank_d" + dim + ".png";
    File.WriteAllBytes(path, h.ToPngGdi());

    var (px, py) = h.ToPixel(0.0, 0.0);
    Console.WriteLine("Griewank dim=" + dim.ToString().PadLeft(2) + " - pixel(0,0) = " + h.Bitmap.GetPixel(px, py) + " (PNG: " + path + ").");
}


Griewank dim= 2 - pixel(0,0) = Color [A=255, R=255, G=4, B=0] (PNG: assets/landscape_multidim_griewank_d2.png).


Griewank dim= 5 - pixel(0,0) = Color [A=255, R=53, G=255, B=0] (PNG: assets/landscape_multidim_griewank_d5.png).


Griewank dim=30 - pixel(0,0) = Color [A=255, R=255, G=0, B=0] (PNG: assets/landscape_multidim_griewank_d30.png).



warning CS1701: En supposant que la référence d'assembly 'System.Drawing.Primitives, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'MetaGeneticSharp.Infrastructure' correspond à l'identité 'System.Drawing.Primitives, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Drawing.Primitives', il se peut que vous deviez fournir une stratégie runtime



## Tell visuel attendu

Pattern transferable L780-L2 ★ `niveau_2d_procedural_dense` pour les **6 nouvelles PNG** :

- **2-D** (Rosenbrock et Griewank) : heatmaps matplotlib-style colorées multi-catégories,
  std élevé sur les 3 canaux R/G/B, palette restreinte selon la fonction.
- **5-D et 30-D** : heatmaps progressivement plus uniformes, gradient de rouge dominant
  (signature L785-L2 ★ inversion chromatique chiffrable `d2 -> d30`).
- **Rosenbrock 30-D** : attendu quasi-uniform-rouge (vallée étroite invisible).
- **Griewank 30-D** : attendu quasi-uniform-orange-clair (produit-cosinus effondré sous MAX).

Stats RGB PIL tell `matplotlib_blanc + niveau_2d_procedural_dense` (L780-L2 ★) ou
`pixel_art_dense` (tell nouveau dérivé de c.732/c.785 = heatmaps haute densité sans fond
blanc dominant — à confirmer après génération effective).

## Exercices

### Exercice 1 : quantifier la *disparition* de la vallée Rosenbrock

Mesurer la **variance spatiale** du canal rouge entre la heatmap Rosenbrock-d2 et
Rosenbrock-d30. Si la vallée disparaît, la variance doit chuter d'un facteur $\geq 5$.
Squelette :

```csharp
var stats = new List<(int dim, double varianceRouge)>();
foreach (int dim in new[] { 2, 5, 30 }) {
    using var h = KnownFunctionLandscape.RenderHeatmap(
        new RosenbrockFitness(), dimension: dim, nbSamples: 50,
        rng: new Random(42), width: 120, height: 120);
    // TODO: collecter les valeurs R des 14400 pixels, calculer variance.
}
// Afficher le ratio variance(d2) / variance(d30).
```

### Exercice 2 : produit-cosinus vs somme-cosinus (Griewank vs Ackley)

Charger Ackley-d2 et Griewank-d2, comparer le **spectre de fréquences spatiales**
(transformée de Fourier 2-D du canal rouge). Le produit de Griewank doit montrer un
spectre **plus épars** (fréquences dictées par $\sqrt{i}$) que la somme d'Ackley
(spectre **continu**).
### Exercice 3 : rudesse du paysage et comptage des minima locaux

Un pixel de la heatmap est un **minimum local** s'il est strictement inférieur
à ses 4 voisins (haut, bas, gauche, droite) sur le canal rouge. Compter le
nombre de minima locaux pour Rosenbrock-d2 et Griewank-d2. Attendu :
Rosenbrock $\approx 1$ minimum (paysage *lisse*, fond de vallée unique) ;
Griewank $\gg 1$ (paysage *rugueux*, le produit de cosinus crée de nombreux
puits). Le ratio quantifie la **rudesse**, la propriété qui rend les
métaheuristiques nécessaires là où un gradient seul reste piégé. Squelette :

```csharp
using var hR = KnownFunctionLandscape.RenderHeatmap(
    new RosenbrockFitness(), dimension: 2, nbSamples: 50,
    rng: new Random(42), width: 120, height: 120);
using var hG = KnownFunctionLandscape.RenderHeatmap(
    new GriewankFitness(), dimension: 2, nbSamples: 50,
    rng: new Random(42), width: 120, height: 120);
// TODO: pour chaque heatmap, parcourir les pixels interieurs (1..118),
// comparer R[pixel] a ses 4 voisins (haut/bas/gauche/droite), compter
// les minima locaux stricts et afficher le ratio Rosenbrock / Griewank.
```

## Liens

- [MGS-7b LandscapeMultidim](MGS-7b-LandscapeMultidim.ipynb) — Ackley dim ∈ {2, 30} (c.785 PR #7993), Rastrigin/Schwefel/Sphere dim ∈ {2, 5, 10, 30} (c.732 PR #7583).
- [MGS-6 Benchmarks](MGS-6-Benchmarks.ipynb) — les 10 fonctions canoniques et leur banc CEC d'origine.
- [MGS-8 LandscapeExplorer](MGS-8-LandscapeExplorer.ipynb) — exploration interactive GTK#.
- Issue [#7483](https://github.com/jsboige/CoursIA/issues/7483) — port de la projection N-D (parent).
- Issue [#7997](https://github.com/jsboige/CoursIA/issues/7997) — c.787 MGS-7c Rosenbrock+Griewank (sujet de ce notebook).
- PR [#7583](https://github.com/jsboige/CoursIA/pull/7583) — port initial MGS-7b N-D.
- PR [#7993](https://github.com/jsboige/CoursIA/pull/7993) — c.785 MGS-7b Ackley.
- L785-L2 ★ — inversion chromatique chiffrable d2 → d30 = signature projection MAX.